# Pandas Basics and Transformation Workflow with IMF WEO Data

This notebook is an introductory pandas lab using `data/WEOApr2026all.xlsx`.

It keeps the steps short and practical: inspect data, clean columns, convert types, handle missing values, filter, sort, create calculated columns, group, reshape, merge, detect outliers, rank, prepare correlations, and export clean data.

## 0. Setup

Run this in a new environment such as Google Colab. If packages already exist, it completes quickly.

In [ ]:
%pip install pandas openpyxl -q

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

## 1. Load the WEO Dataset

The normal course path is `data/WEOApr2026all.xlsx`. If you move this notebook to Colab, upload `WEOApr2026all.xlsx` into `/content/`.

If the Excel file is not available, a small fallback dataset is created so the pandas examples still run.

In [ ]:
course_path = Path("data/WEOApr2026all.xlsx")
colab_path = Path("/content/WEOApr2026all.xlsx")
excel_path = course_path if course_path.exists() else colab_path if colab_path.exists() else None

if excel_path:
    df = pd.read_excel(excel_path, sheet_name="Countries")
    print(f"Loaded WEO file: {excel_path}")
else:
    print("WEO file not found. Using small fallback data.")
    df = pd.DataFrame({
        "COUNTRY.ID": ["ZWE", "ZAF", "BWA", "LSO", "ZWE"],
        "COUNTRY": ["Zimbabwe", "South Africa", "Botswana", "Lesotho", "Zimbabwe"],
        "INDICATOR.ID": ["PCPIPCH", "PCPIPCH", "PCPIPCH", "PCPIPCH", "NGDP_RPCH"],
        "INDICATOR": ["Inflation, average consumer prices", "Inflation, average consumer prices", "Inflation, average consumer prices", "Inflation, average consumer prices", "Gross domestic product (GDP), Constant prices, Percent change"],
        "UNIT": ["Percent", "Percent", "Percent", "Percent", "Percent"],
        2020: [557.2, 3.2, 1.9, 5.0, -7.8],
        2021: [98.5, 4.6, 6.7, 6.0, 8.5],
        2022: [193.4, 6.9, 12.2, np.nan, 6.1],
        2023: [667.4, 6.0, 5.1, 6.4, 5.3],
        2024: [57.5, 4.4, 2.8, 6.1, 2.0],
        2025: [21.0, 4.0, 3.5, 5.8, 3.2],
        2026: [10.0, 4.5, 4.0, 5.5, 3.0]
    })

df.head()

## 2. View and Understand the DataFrame

Before cleaning data, inspect it. These are the first commands to teach with any new dataset.

In [ ]:
display(df.head())
display(df.tail())
print("Shape:", df.shape)
print("Columns:", list(df.columns[:15]))
df.info()

In [ ]:
df.describe(include="all").T.head(12)

## 3. Select Columns and Rows

Use `[]` for columns, `iloc` for row positions, and conditions for filtering.

In [ ]:
display(df["COUNTRY"].head())
display(df[["COUNTRY", "INDICATOR", 2024, 2025, 2026]].head())
display(df.iloc[0:5])
display(df[df["COUNTRY"] == "Zimbabwe"].head())

## 4. Clean and Rename Columns

Raw column names may contain spaces, dots, uppercase text, and mixed data types. Clean them once so later code is easier.

In [ ]:
clean_df = df.copy()
clean_df.columns = (
    clean_df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(".", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

clean_df = clean_df.rename(columns={
    "country_id": "country_code",
    "indicator_id": "indicator_code"
})

clean_df.columns[:20]

## 5. Check and Convert Data Types

Year columns must be numeric before calculations. `errors="coerce"` turns invalid values into `NaN`.

In [ ]:
year_cols = [str(year) for year in range(2020, 2027) if str(year) in clean_df.columns]
print(year_cols)

display(clean_df[year_cols].dtypes)
clean_df[year_cols] = clean_df[year_cols].apply(pd.to_numeric, errors="coerce")
display(clean_df[year_cols].dtypes)

## 6. Handle Missing Values and Duplicates

Always measure missing values and duplicates before deciding how to clean them.

In [ ]:
display(clean_df[["country", "indicator"] + year_cols].isna().sum())
print("Duplicate rows:", clean_df.duplicated().sum())

clean_df = clean_df.drop_duplicates()

# Teaching example: create a filled copy. In production, document the fill rule.
filled_df = clean_df.copy()
filled_df[year_cols] = filled_df[year_cols].fillna(filled_df[year_cols].median(numeric_only=True))
filled_df[year_cols].isna().sum()

## 7. Filter, Sort, and Create Calculated Columns

This section focuses on inflation indicators for Southern African countries.

In [ ]:
southern_africa = ["Zimbabwe", "South Africa", "Botswana", "Lesotho", "Lesotho, Kingdom of", "Namibia", "Zambia"]

inflation_df = clean_df[
    clean_df["country"].isin(southern_africa)
    & (clean_df["indicator_code"] == "PCPIPCH")
].copy()

inflation_df["change_2025_2026"] = inflation_df["2026"] - inflation_df["2025"]
inflation_df["pct_change_2025_2026"] = ((inflation_df["2026"] - inflation_df["2025"]) / inflation_df["2025"]) * 100
inflation_df["recent_average"] = inflation_df[["2024", "2025", "2026"]].mean(axis=1)
inflation_df["recent_max"] = inflation_df[["2024", "2025", "2026"]].max(axis=1)
inflation_df["recent_min"] = inflation_df[["2024", "2025", "2026"]].min(axis=1)

inflation_df.sort_values("2026", ascending=False)[["country", "2024", "2025", "2026", "change_2025_2026", "recent_average"]]

## 8. Apply Functions and Clean Strings

`apply()` is useful for custom categories. String methods clean and standardise text.

In [ ]:
def inflation_category(value):
    if pd.isna(value):
        return "Missing"
    if value < 5:
        return "Low"
    if value < 15:
        return "Moderate"
    return "High"

inflation_df["country_clean"] = inflation_df["country"].str.strip()
inflation_df["country_upper"] = inflation_df["country_clean"].str.upper()
inflation_df["inflation_category"] = inflation_df["2026"].apply(inflation_category)

inflation_df[["country_clean", "country_upper", "2026", "inflation_category"]]

## 9. Group and Aggregate

`groupby()` turns detail rows into summary tables.

In [ ]:
inflation_df.groupby("inflation_category")["2026"].agg(["count", "mean", "median", "min", "max"])

## 10. Reshape Data: Wide to Long and Long to Wide

WEO is wide because each year is a column. For database loading and many analytics tasks, long format is better.

In [ ]:
long_df = inflation_df.melt(
    id_vars=["country_code", "country", "indicator_code", "indicator", "unit"],
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)
long_df["year"] = long_df["year"].astype(int)
display(long_df.head())

wide_again = long_df.pivot_table(
    index=["country", "indicator"],
    columns="year",
    values="value",
    aggfunc="mean"
).reset_index()
wide_again.head()

## 11. Merge and Concatenate DataFrames

`merge()` joins related data. `concat()` stacks similar datasets.

In [ ]:
region_lookup = pd.DataFrame({
    "country": ["Zimbabwe", "South Africa", "Botswana", "Lesotho", "Namibia", "Zambia"],
    "region": ["Southern Africa"] * 6
})

merged_df = long_df.merge(region_lookup, on="country", how="left")
recent_actuals = long_df[long_df["year"].isin([2024, 2025])]
forecast_values = long_df[long_df["year"].isin([2026])]
combined_recent = pd.concat([recent_actuals, forecast_values], ignore_index=True)

display(merged_df[["country", "region", "year", "value"]].head())
display(combined_recent[["country", "year", "value"]].head())

## 12. Outliers, Categories, Ranking, and Correlation

These steps prepare data for deeper analysis.

In [ ]:
q1 = inflation_df["2026"].quantile(0.25)
q3 = inflation_df["2026"].quantile(0.75)
iqr = q3 - q1
outliers = inflation_df[(inflation_df["2026"] < q1 - 1.5 * iqr) | (inflation_df["2026"] > q3 + 1.5 * iqr)]

inflation_df["2026_capped"] = inflation_df["2026"].clip(
    inflation_df["2026"].quantile(0.05),
    inflation_df["2026"].quantile(0.95)
)

inflation_df["inflation_level"] = pd.cut(
    inflation_df["2026"],
    bins=[-np.inf, 5, 15, np.inf],
    labels=["Low", "Moderate", "High"]
)
inflation_df["rank_2026"] = inflation_df["2026"].rank(ascending=False)

display(outliers[["country", "2026"]])
display(inflation_df.sort_values("rank_2026")[["country", "2026", "inflation_level", "rank_2026"]])
inflation_df[year_cols + ["change_2025_2026", "recent_average"]].corr(numeric_only=True)

## 13. Export Clean Data

Save the final outputs for reporting, database loading, or later analysis.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

analysis_ready_path = output_dir / "weo_inflation_analysis_ready.csv"
long_path = output_dir / "weo_inflation_long.csv"

inflation_df.to_csv(analysis_ready_path, index=False)
long_df.to_csv(long_path, index=False)

analysis_ready_path, long_path

## Quick Reference

| Technique | Pandas command |
|---|---|
| View data | `head()`, `tail()`, `info()`, `describe()` |
| Select data | `[]`, `iloc`, boolean filtering |
| Rename columns | `rename()` |
| Clean columns | `str.strip()`, `str.lower()`, `str.replace()` |
| Convert data types | `pd.to_numeric()`, `astype()` |
| Missing values | `isna()`, `fillna()`, `dropna()` |
| Duplicates | `drop_duplicates()` |
| Sort rows | `sort_values()` |
| New columns | assignments and calculations |
| Custom logic | `apply()` |
| Group data | `groupby()`, `agg()` |
| Wide to long | `melt()` |
| Long to wide | `pivot_table()` |
| Combine data | `merge()`, `concat()` |
| Outliers | `quantile()`, `clip()` |
| Ranking | `rank()` |
| Export | `to_csv()`, `to_excel()` |
